# Set up library

In [ ]:
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
from pathlib import Path
import h5py
import os
import librosa
import json
import unicodedata
import pandas as pd






# Helpful function

In [ ]:
hop_length = 512
n_bins=168
sr = 48000
bins_per_octave=24
fmin=librosa.note_to_hz('C3')

## get cqt

In [ ]:
def get_CQT_of_a_audio(y, sr):

    C = librosa.cqt(
        y,
        sr=sr,
        hop_length=hop_length,
        fmin=fmin,
        n_bins=n_bins,
        bins_per_octave=bins_per_octave
    )

    return C

## Helper to create csv file

In [ ]:
def create_label_template(audio_path, output_csv, hop_length=512):

    y, sr = librosa.load(audio_path, sr=sr)

    duration = librosa.get_duration(y=y, sr=sr)

    frame_time = hop_length / sr

    total_frames = int(duration / frame_time)

    frames = np.arange(total_frames)

    start_times = frames * frame_time
    end_times = start_times + frame_time

    df = pd.DataFrame({
        "frame": frames,
        "start_time": start_times,
        "end_time": end_times,
        "label": "Default"
    })

    df.to_csv(output_csv, index=False)

def create_labels_for_audios_in_folder(audio_folder, audio_folder_name = "audio", csv_folder_name = "frame_labels"):


    if os.path.exists(audio_folder):
        for root, dirs, files in os.walk(audio_folder):
            for file in files:
                if file.endswith(".wav"):
                    csv_root = root.replace(audio_folder_name, csv_folder_name)
                if os.path.exists(csv_root):
                    csv_file_name = file.replace(".wav", ".csv")
                    csv_path = os.path.join(csv_root, csv_file_name)
                    audio_path = os.path.join(root, file)
                    create_label_template(audio_path, csv_path)


                else:
                    print(csv_root, "doesnt exists !")

    else:
        print("Cant find folder at", audio_folder)


## helper to create spectrogrames of audio

In [ ]:
def analyze_spectrogram_by_CQT_of_audio(audio_path, img_path, is_saving = True):

    C, sr = get_CQT_of_a_audio(audio_path)

    n_frames = C.shape[1]

    frame_time = hop_length / sr

    times = np.arange(n_frames) * frame_time


    C_db = librosa.amplitude_to_db(np.abs(C))

    plt.figure(figsize=(12,6))

    librosa.display.specshow(
            C_db,
            sr=sr,
            hop_length=hop_length,
            x_axis='time',
            y_axis='cqt_note'
    )

    plt.title(audio_path)

    for t in times:
        plt.axvline(x=t, color='white', alpha = 0.5, linewidth=1)

    step = max(1, n_frames // 20)

    frame_ticks = np.arange(0, n_frames, step)
    frame_tick_times = frame_ticks * frame_time

    plt.xticks(frame_tick_times, frame_ticks)

    plt.xlabel("Frame index")

    plt.colorbar(format="%+2.0f dB")

    plt.tight_layout()

    print("Show!")

    plt.show()

    if is_saving:
       plt.savefig(img_path, dpi=300)

In [ ]:
def analyzing_spectrograms_for_audios_in_folder(audio_folder):

    audio_folder_name = "audio"
    img_folder_name = "spectrograms"

    if os.path.exists(audio_folder):
        for root, dirs, files in os.walk(audio_folder):
            for file in files:
                if file.endswith(".wav"):
                    img_root = root.replace(audio_folder_name, img_folder_name)
                if os.path.exists(img_root):
                    img_file_name = file.replace(".wav", ".png")
                    img_path = os.path.join(img_root, img_file_name)
                    audio_path = os.path.join(root, file)
                    analyze_spectrogram_by_CQT_of_audio(audio_path, img_path)


                else:
                    print(img_root, "doesnt exists !")

    else:
        print("Cant find folder at", audio_folder)

## Create Csv file

In [ ]:
audio_folder = "./Data/audio_extends/Khèn 8/Đa_ống/3_ống"
create_labels_for_audios_in_folder(audio_folder, audio_folder_name="audio_extends")